In [1]:
#!pip install -qqq arize-otel

In [2]:
import os
space_id = os.environ["ARIZE_SPACE_ID"]
api_key = os.environ["ARIZE_API_KEY"]

In [3]:
project_name = "nlp_to_sql_test1"

In [4]:
# Import open-telemetry dependencies
from arize.otel import register
import pandas as pd
import json
import uuid
import random
from opentelemetry import trace
from opentelemetry.trace import SpanKind, Status, StatusCode

tracer_provider = register(
    space_id=space_id,
    api_key=api_key,
    project_name=project_name,
)

🔭 OpenTelemetry Tracing Details 🔭
|  Arize Project: nlp_to_sql_test1
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: otlp.arize.com
|  Transport: gRPC
|  Transport Headers: {'authorization': '****', 'api_key': '****', 'arize-space-id': '****', 'space_id': '****', 'arize-interface': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [5]:
# Config: model/region placeholders for LLM span attributes (no real API calls)
MODEL_REGION = "us-east-1"
MODEL_ID = "claude-3-7-sonnet-latest"

In [6]:
# Synthetic index: fields (field_id, name, type, description)
INDEX_FIELDS = [
    {"field_id": "f1", "name": "genre", "type": "string", "description": "Content genre or category"},
    {"field_id": "f2", "name": "year", "type": "integer", "description": "Release or publication year"},
    {"field_id": "f3", "name": "rating", "type": "string", "description": "Content rating or maturity"},
    {"field_id": "f4", "name": "language", "type": "string", "description": "Primary language"},
    {"field_id": "f5", "name": "duration_min", "type": "integer", "description": "Duration in minutes"},
    {"field_id": "f6", "name": "country", "type": "string", "description": "Country or region of origin"},
]

In [7]:
# Controlled vocabulary: cv_id, display_name, field_id
CONTROLLED_VOCABULARY = [
    {"cv_id": "cv1", "display_name": "Comedy", "field_id": "f1"},
    {"cv_id": "cv2", "display_name": "Drama", "field_id": "f1"},
    {"cv_id": "cv3", "display_name": "Sci-Fi", "field_id": "f1"},
    {"cv_id": "cv4", "display_name": "PG-13", "field_id": "f3"},
    {"cv_id": "cv5", "display_name": "R", "field_id": "f3"},
    {"cv_id": "cv6", "display_name": "English", "field_id": "f4"},
    {"cv_id": "cv7", "display_name": "Spanish", "field_id": "f4"},
    {"cv_id": "cv8", "display_name": "United States", "field_id": "f6"},
]

In [8]:
# Scenarios: query, field_rag_ids, cv_rag_ids, generated_filter, validation_passed, retry_used, use_entity_mention, search_results
SCENARIOS = [
    {
        "scenario_id": 0,
        "query": "I want to see all movies from the 90s about robots from the US",
        "field_rag_ids": ["f1", "f2", "f6"],
        "cv_rag_ids": ["cv3", "cv8"],
        "generated_filter": 'genre eq "Sci-Fi" and year >= 1990 and year < 2000 and country eq "United States"',
        "validation_passed": True,
        "retry_used": False,
        "use_entity_mention": False,
        "search_results": {"result_count": 12, "document_ids": ["doc_001", "doc_002", "doc_003", "doc_004", "doc_005"]},
    },
    {
        "scenario_id": 1,
        "query": "Show me every drama from the last five years that's PG-13 and in English",
        "field_rag_ids": ["f1", "f2", "f3", "f4"],
        "cv_rag_ids": ["cv2", "cv4", "cv6"],
        "generated_filter": 'genre eq "Drama" and year >= 2019 and rating eq "PG-13" and language eq "English"',
        "validation_passed": True,
        "retry_used": False,
        "use_entity_mention": False,
        "search_results": {"result_count": 28, "document_ids": ["doc_101", "doc_102", "doc_103", "doc_104"]},
    },
    {
        "scenario_id": 2,
        "query": "Find me Spanish-language comedies that are under two hours",
        "field_rag_ids": ["f1", "f4", "f5"],
        "cv_rag_ids": ["cv1", "cv7"],
        "generated_filter": 'genre eq "Comedy" and language eq "Spanish" and duration_min < 120',
        "validation_passed": True,
        "retry_used": False,
        "use_entity_mention": False,
        "search_results": {"result_count": 7, "document_ids": ["doc_201", "doc_202", "doc_203"]},
    },
    {
        "scenario_id": 3,
        "query": "I need a list of all content that's in Spanish, any genre",
        "field_rag_ids": ["f4"],
        "cv_rag_ids": ["cv7"],
        "generated_filter": 'language eq "Spanish"',
        "validation_passed": True,
        "retry_used": False,
        "use_entity_mention": False,
        "search_results": {"result_count": 45, "document_ids": ["doc_301", "doc_302", "doc_303", "doc_304", "doc_305"]},
    },
    {
        "scenario_id": 4,
        "query": "R-rated dramas that run over 90 minutes, from any country",
        "field_rag_ids": ["f1", "f3", "f5"],
        "cv_rag_ids": ["cv2", "cv5"],
        "generated_filter": 'genre eq "Drama" and rating eq "R" and duration_min > 90',
        "validation_passed": True,
        "retry_used": False,
        "use_entity_mention": False,
        "search_results": {"result_count": 19, "document_ids": ["doc_401", "doc_402", "doc_403"]},
    },
    {
        "scenario_id": 5,
        "query": "Just comedies @genre",
        "field_rag_ids": ["f1"],
        "cv_rag_ids": ["cv1"],
        "generated_filter": 'genre eq "Comedy"',
        "validation_passed": True,
        "retry_used": False,
        "use_entity_mention": True,
        "search_results": {"result_count": 62, "document_ids": ["doc_501", "doc_502", "doc_503", "doc_504"]},
    },
    {
        "scenario_id": 6,
        "query": "I want comedies from 2020 that are in English",
        "field_rag_ids": ["f1", "f2", "f4"],
        "cv_rag_ids": ["cv1", "cv6"],
        "generated_filter": 'genre eq "Comedy" year eq 2020',  # invalid: missing "and"
        "validation_passed": False,
        "retry_used": True,
        "generated_filter_retry": 'genre eq "Comedy" and year eq 2020 and language eq "English"',
        "use_entity_mention": False,
        "search_results": {"result_count": 8, "document_ids": ["doc_601", "doc_602", "doc_603"]},
    },
    {
        "scenario_id": 7,
        "query": "Show me English-language comedies from the last few years",
        "field_rag_ids": ["f1", "f4", "f2"],
        "cv_rag_ids": ["cv1", "cv6"],
        "generated_filter": "invalid dsl here",
        "validation_passed": False,
        "retry_used": True,
        "generated_filter_retry": 'genre eq "Comedy" and language eq "English" and year >= 2021',
        "use_entity_mention": False,
        "search_results": {"result_count": 15, "document_ids": ["doc_701", "doc_702", "doc_703", "doc_704"]},
    },
]

In [9]:
def get_nl_to_filter_data(scenario_id=None):
    """Get data for one NL-to-filter trace. scenario_id: 0..len(SCENARIOS)-1 or None for random."""
    if scenario_id is None:
        scenario_id = random.randint(0, len(SCENARIOS) - 1)
    idx = scenario_id % len(SCENARIOS)
    s = SCENARIOS[idx]
    field_docs = [f for f in INDEX_FIELDS if f["field_id"] in s["field_rag_ids"]]
    cv_docs = [c for c in CONTROLLED_VOCABULARY if c["cv_id"] in s["cv_rag_ids"]]
    return {
        "scenario_id": idx,
        "query": s["query"],
        "field_docs": field_docs,
        "cv_docs": cv_docs,
        "field_rag_ids": s["field_rag_ids"],
        "cv_rag_ids": s["cv_rag_ids"],
        "generated_filter": s["generated_filter"],
        "validation_passed": s["validation_passed"],
        "retry_used": s["retry_used"],
        "use_entity_mention": s.get("use_entity_mention", False),
        "generated_filter_retry": s.get("generated_filter_retry"),
        "search_results": s.get("search_results", {"result_count": 0, "document_ids": []}),
    }

In [10]:
# Trace: CHAIN -> AGENT context_builder -> RETRIEVER field_rag, cv_rag -> AGENT filter_orchestrator -> LLM generate_filter -> TOOL validate_dsl [-> LLM generate_filter_retry] -> TOOL execute_search
from opentelemetry.trace.status import Status, StatusCode
import time

def create_nl_to_filter_trace(tracer, scenario_id=None, session_id=None):
    """Create one Natural Language to Filter pipeline trace. Returns span_data dict."""
    data = get_nl_to_filter_data(scenario_id)
    query = data["query"]
    field_docs = data["field_docs"]
    cv_docs = data["cv_docs"]
    generated_filter = data["generated_filter"]
    validation_passed = data["validation_passed"]
    retry_used = data["retry_used"]
    use_entity_mention = data["use_entity_mention"]
    generated_filter_retry = data.get("generated_filter_retry")
    search_results = data.get("search_results", {"result_count": 0, "document_ids": []})

    # Inflated synthetic latencies (ms)
    context_builder_latency_ms = random.uniform(80, 400)
    field_rag_latency_ms = random.uniform(120, 500)
    cv_rag_latency_ms = random.uniform(120, 500)
    filter_orchestrator_latency_ms = random.uniform(60, 300)
    llm_latency_ms = random.uniform(1800, 4500)
    validate_dsl_latency_ms = random.uniform(30, 180)
    llm_retry_latency_ms = random.uniform(1500, 4000) if retry_used else 0
    execute_search_latency_ms = random.uniform(100, 600)
    chain_overhead_ms = random.uniform(20, 100)
    chain_latency_ms = (
        context_builder_latency_ms + field_rag_latency_ms + cv_rag_latency_ms
        + filter_orchestrator_latency_ms + llm_latency_ms + validate_dsl_latency_ms
        + llm_retry_latency_ms + execute_search_latency_ms + chain_overhead_ms
    )
    prompt_tokens = random.randint(300, 800)
    completion_tokens = random.randint(50, 200)
    retry_prompt_tokens = random.randint(280, 750)
    retry_completion_tokens = random.randint(45, 180)

    chain_name = "Natural Language to Filter Pipeline"
    span_ids = {}

    with tracer.start_as_current_span(chain_name) as chain_span:
        chain_span.set_attribute("openinference.span.kind", "CHAIN")
        chain_span.set_attribute("input.value", query)
        chain_span.set_attribute("input.mime_type", "text/plain")
        if session_id:
            chain_span.set_attribute("session.id", session_id)
        span_ids["chain_span_id"] = format(chain_span.get_span_context().span_id, "016x")

        with tracer.start_as_current_span("context_builder") as ctx_agent_span:
            ctx_agent_span.set_attribute("openinference.span.kind", "AGENT")
            ctx_agent_span.set_attribute("agent.name", "context_builder")
            ctx_agent_span.set_attribute("input.value", json.dumps({"query": query}))
            ctx_agent_span.set_attribute("latency_ms", round(context_builder_latency_ms, 2))
            span_ids["context_builder_span_id"] = format(ctx_agent_span.get_span_context().span_id, "016x")

            if not use_entity_mention or len(field_docs) > 0:
                with tracer.start_as_current_span("field_rag") as field_rag_span:
                    field_rag_span.set_attribute("openinference.span.kind", "RETRIEVER")
                    field_rag_span.set_attribute("input.value", json.dumps({"query": query}))
                    field_rag_span.set_attribute("output.value", json.dumps([{"field_id": f["field_id"], "name": f["name"]} for f in field_docs]))
                    field_rag_span.set_attribute("latency_ms", round(field_rag_latency_ms, 2))
                    span_ids["field_rag_span_id"] = format(field_rag_span.get_span_context().span_id, "016x")

            with tracer.start_as_current_span("cv_rag") as cv_rag_span:
                cv_rag_span.set_attribute("openinference.span.kind", "RETRIEVER")
                cv_rag_span.set_attribute("input.value", json.dumps({"query": query}))
                cv_rag_span.set_attribute("output.value", json.dumps([{"cv_id": c["cv_id"], "display_name": c["display_name"]} for c in cv_docs]))
                cv_rag_span.set_attribute("latency_ms", round(cv_rag_latency_ms, 2))
                span_ids["cv_rag_span_id"] = format(cv_rag_span.get_span_context().span_id, "016x")

        with tracer.start_as_current_span("filter_orchestrator") as orch_agent_span:
            orch_agent_span.set_attribute("openinference.span.kind", "AGENT")
            orch_agent_span.set_attribute("agent.name", "filter_orchestrator")
            orch_agent_span.set_attribute("input.value", json.dumps({"query": query, "field_ids": data["field_rag_ids"], "cv_ids": data["cv_rag_ids"]}))
            orch_agent_span.set_attribute("latency_ms", round(filter_orchestrator_latency_ms, 2))
            span_ids["filter_orchestrator_span_id"] = format(orch_agent_span.get_span_context().span_id, "016x")

            with tracer.start_as_current_span("generate_filter") as llm_span:
                llm_span.set_attribute("openinference.span.kind", "LLM")
                llm_span.set_attribute("llm.model_name", MODEL_ID)
                llm_span.set_attribute("llm.provider", "anthropic")
                llm_span.set_attribute("llm.token_count.prompt", prompt_tokens)
                llm_span.set_attribute("llm.token_count.completion", completion_tokens)
                llm_span.set_attribute("llm.token_count.total", prompt_tokens + completion_tokens)
                llm_span.set_attribute("input.value", query)
                llm_span.set_attribute("output.value", generated_filter)
                llm_span.set_attribute("latency_ms", round(llm_latency_ms, 2))
                span_ids["generate_filter_span_id"] = format(llm_span.get_span_context().span_id, "016x")

            with tracer.start_as_current_span("validate_dsl") as tool_span:
                tool_span.set_attribute("openinference.span.kind", "TOOL")
                tool_span.set_attribute("input.value", generated_filter)
                tool_span.set_attribute("output.value", json.dumps({"valid": validation_passed}))
                tool_span.set_attribute("latency_ms", round(validate_dsl_latency_ms, 2))
                span_ids["validate_dsl_span_id"] = format(tool_span.get_span_context().span_id, "016x")

            if retry_used and generated_filter_retry:
                with tracer.start_as_current_span("generate_filter_retry") as retry_llm_span:
                    retry_llm_span.set_attribute("openinference.span.kind", "LLM")
                    retry_llm_span.set_attribute("llm.model_name", MODEL_ID)
                    retry_llm_span.set_attribute("llm.token_count.prompt", retry_prompt_tokens)
                    retry_llm_span.set_attribute("llm.token_count.completion", retry_completion_tokens)
                    retry_llm_span.set_attribute("llm.token_count.total", retry_prompt_tokens + retry_completion_tokens)
                    retry_llm_span.set_attribute("input.value", query)
                    retry_llm_span.set_attribute("output.value", generated_filter_retry)
                    retry_llm_span.set_attribute("latency_ms", round(llm_retry_latency_ms, 2))
                    span_ids["generate_filter_retry_span_id"] = format(retry_llm_span.get_span_context().span_id, "016x")
                final_filter = generated_filter_retry
            else:
                final_filter = generated_filter

            search_output = json.dumps({
                "result_count": search_results.get("result_count", 0),
                "document_ids": search_results.get("document_ids", []),
                "summary": f"Found {search_results.get('result_count', 0)} results",
            })
            with tracer.start_as_current_span("execute_search") as exec_span:
                exec_span.set_attribute("openinference.span.kind", "TOOL")
                exec_span.set_attribute("input.value", final_filter)
                exec_span.set_attribute("output.value", search_output)
                exec_span.set_attribute("latency_ms", round(execute_search_latency_ms, 2))
                span_ids["execute_search_span_id"] = format(exec_span.get_span_context().span_id, "016x")

            orch_agent_span.set_attribute("output.value", search_output)
        chain_span.set_attribute("output.value", search_output)
        chain_span.set_attribute("latency_ms", round(chain_latency_ms, 2))
        chain_span.set_status(Status(StatusCode.OK))

    # Final pipeline success: passed on first try, or after retry
    final_validation_passed = validation_passed or (retry_used and bool(generated_filter_retry))

    return {
        "span_data": {
            **span_ids,
            "query": query,
            "generated_filter": final_filter,
            "validation_passed": final_validation_passed,
            "first_validation_passed": validation_passed,
            "retry_used": retry_used,
            "session_id": session_id,
            "scenario_id": data["scenario_id"],
        }
    }

In [11]:
# Session evaluations: root span of first trace per session — session_eval.PipelineSuccess.* (Arize convention)
def generate_session_evaluations_from_chain_spans(span_df: pd.DataFrame) -> pd.DataFrame:
    """Build DataFrame with one row per session: context.span_id = root span of first trace, session_eval.PipelineSuccess.*"""
    chain_spans = span_df[span_df["openinference.span.kind"] == "CHAIN"].copy()
    if len(chain_spans) == 0:
        return pd.DataFrame(columns=["context.span_id", "session_eval.PipelineSuccess.label", "session_eval.PipelineSuccess.score", "session_eval.PipelineSuccess.explanation"])
    if "session_id" not in chain_spans.columns:
        return pd.DataFrame(columns=["context.span_id", "session_eval.PipelineSuccess.label", "session_eval.PipelineSuccess.score", "session_eval.PipelineSuccess.explanation"])
    chain_spans = chain_spans[chain_spans["session_id"].notna()]
    if len(chain_spans) == 0:
        return pd.DataFrame(columns=["context.span_id", "session_eval.PipelineSuccess.label", "session_eval.PipelineSuccess.score", "session_eval.PipelineSuccess.explanation"])
    if "timestamp" not in chain_spans.columns:
        chain_spans["timestamp"] = range(len(chain_spans))
    first_per_session = chain_spans.sort_values("timestamp").groupby("session_id").first().reset_index()
    rows = []
    for _, row in first_per_session.iterrows():
        span_id = row.get("context.span_id", "")
        passed = row.get("validation_passed", True)
        label = "pass" if passed else "fail"
        score = 0 if passed else 1
        explanation = "Pipeline succeeded: filter generated and validated." if passed else "Pipeline failed: validation or retry path."
        rows.append({
            "context.span_id": span_id,
            "session_eval.PipelineSuccess.label": label,
            "session_eval.PipelineSuccess.score": score,
            "session_eval.PipelineSuccess.explanation": explanation,
        })
    return pd.DataFrame(rows)

In [12]:
# Trace evals: separate DataFrames per eval type (only rows with real label; no n/a)
def create_evaluations_from_span_data(span_df: pd.DataFrame):
    """Build separate DataFrames per eval type. Only rows where that eval applies; only that eval's columns."""
    context_relevance_rows = []
    syntactic_rows = []
    semantic_rows = []
    validation_rows = []
    agent_trajectory_rows = []

    for _, row in span_df.iterrows():
        kind = row.get("openinference.span.kind", "")
        span_id = row.get("context.span_id", "")

        if kind == "RETRIEVER":
            label = "pass" if random.random() < 0.88 else "fail"
            context_relevance_rows.append({
                "context.span_id": span_id,
                "eval.ContextRelevance.label": label,
                "eval.ContextRelevance.score": 0 if label == "pass" else 1,
                "eval.ContextRelevance.explanation": "Retrieved fields/vocabulary matched the query." if label == "pass" else "Context relevance low.",
            })
        elif kind == "LLM":
            syn_label = "pass" if row.get("validation_passed", True) or row.get("retry_used") else "fail"
            syntactic_rows.append({
                "context.span_id": span_id,
                "eval.SyntacticCorrectness.label": syn_label,
                "eval.SyntacticCorrectness.score": 0 if syn_label == "pass" else 1,
                "eval.SyntacticCorrectness.explanation": "Generated filter parses." if syn_label == "pass" else "Generated filter did not parse.",
            })
            sem_label = "pass" if random.random() < 0.9 else "fail"
            semantic_rows.append({
                "context.span_id": span_id,
                "eval.SemanticCorrectness.label": sem_label,
                "eval.SemanticCorrectness.score": 0 if sem_label == "pass" else 1,
                "eval.SemanticCorrectness.explanation": "No hallucinated index fields or vocabulary values." if sem_label == "pass" else "Hallucinated fields or values.",
            })
        elif kind == "TOOL":
            if row.get("tool_name") == "execute_search":
                continue
            passed = row.get("first_validation_passed", row.get("validation_passed", True))
            validation_rows.append({
                "context.span_id": span_id,
                "eval.ValidationOutcome.label": "pass" if passed else "fail",
                "eval.ValidationOutcome.score": 0 if passed else 1,
                "eval.ValidationOutcome.explanation": "Validation passed." if passed else "Validation failed.",
            })
        elif kind == "AGENT":
            label = "pass" if random.random() < 0.85 else "fail"
            agent_trajectory_rows.append({
                "context.span_id": span_id,
                "trace_eval.AgentTrajectory.label": label,
                "trace_eval.AgentTrajectory.score": 0 if label == "pass" else 1,
                "trace_eval.AgentTrajectory.explanation": "Orchestration correct." if label == "pass" else "Orchestration deviation.",
            })

    return {
        "context_relevance": pd.DataFrame(context_relevance_rows),
        "syntactic_correctness": pd.DataFrame(syntactic_rows),
        "semantic_correctness": pd.DataFrame(semantic_rows),
        "validation_outcome": pd.DataFrame(validation_rows),
        "agent_trajectory": pd.DataFrame(agent_trajectory_rows),
    }

# Single NL-to-Filter Trace Test

In [13]:
# Test one trace
tracer = trace.get_tracer(__name__)
result = create_nl_to_filter_trace(tracer=tracer, scenario_id=0, session_id="test_session_nl_001")
sd = result["span_data"]
print(f"Query: {sd['query']}")
print(f"Generated filter: {sd['generated_filter']}, validation_passed={sd['validation_passed']}, retry_used={sd['retry_used']}")
print(f"Chain: {sd['chain_span_id']}, context_builder: {sd.get('context_builder_span_id')}, field_rag: {sd.get('field_rag_span_id')}, cv_rag: {sd.get('cv_rag_span_id')}, execute_search: {sd.get('execute_search_span_id')}")
trace.get_tracer_provider().force_flush(timeout_millis=30000)
print("Single trace exported.")

Query: I want to see all movies from the 90s about robots from the US
Generated filter: genre eq "Sci-Fi" and year >= 1990 and year < 2000 and country eq "United States", validation_passed=True, retry_used=False
Chain: eb0fc327a55253a7, total spans: 19 (target ~20)
Single trace exported.


# Batch NL-to-Filter trace creation

In [14]:
# Batch: N traces, random scenario_id and session_id; collect span metadata for evals
NUM_TRACES = 400
CHUNK_SIZE = 500
tracer = trace.get_tracer(__name__)
SESSION_SIZE_MIN, SESSION_SIZE_MAX = 2, 5
scenario_ids = list(range(len(SCENARIOS)))

span_data_list = []
current_session_id = None
traces_in_session = 0
session_size = 0
timestamp = 0

for i in range(NUM_TRACES):
    if current_session_id is None or traces_in_session >= session_size:
        current_session_id = f"session_{random.randint(100000, 999999)}"
        session_size = random.randint(SESSION_SIZE_MIN, SESSION_SIZE_MAX)
        traces_in_session = 0
    scenario_id = random.choice(scenario_ids)
    result = create_nl_to_filter_trace(tracer=tracer, scenario_id=scenario_id, session_id=current_session_id)
    traces_in_session += 1
    timestamp += 1
    if "span_data" not in result:
        continue
    sd = result["span_data"]
    base = {"query": sd["query"], "generated_filter": sd["generated_filter"], "validation_passed": sd["validation_passed"], "first_validation_passed": sd.get("first_validation_passed", sd["validation_passed"]), "retry_used": sd["retry_used"], "timestamp": timestamp}

    span_data_list.append({"context.span_id": sd["chain_span_id"], "openinference.span.kind": "CHAIN", "session_id": sd.get("session_id"), **base})
    span_data_list.append({"context.span_id": sd["context_builder_span_id"], "openinference.span.kind": "AGENT", **base})
    if sd.get("field_rag_span_id"):
        span_data_list.append({"context.span_id": sd["field_rag_span_id"], "openinference.span.kind": "RETRIEVER", **base})
    span_data_list.append({"context.span_id": sd["cv_rag_span_id"], "openinference.span.kind": "RETRIEVER", **base})
    span_data_list.append({"context.span_id": sd["filter_orchestrator_span_id"], "openinference.span.kind": "AGENT", **base})
    span_data_list.append({"context.span_id": sd["generate_filter_span_id"], "openinference.span.kind": "LLM", **base})
    span_data_list.append({"context.span_id": sd["validate_dsl_span_id"], "openinference.span.kind": "TOOL", "tool_name": "validate_dsl", **base})
    if sd.get("generate_filter_retry_span_id"):
        span_data_list.append({"context.span_id": sd["generate_filter_retry_span_id"], "openinference.span.kind": "LLM", **base})
    span_data_list.append({"context.span_id": sd["execute_search_span_id"], "openinference.span.kind": "TOOL", "tool_name": "execute_search", **base})

flush_success = trace.get_tracer_provider().force_flush(timeout_millis=60000)
collected_span_data = span_data_list
print(f"Created {NUM_TRACES} traces. Collected {len(span_data_list)} span records. Flush: {flush_success}")

Created 400 traces. Collected 7688 span records. Flush: True


In [15]:
# Evaluation generation and logging: one DataFrame per eval type, only rows with real labels, chunk 500
if "collected_span_data" not in globals() or not collected_span_data:
    print("Run the batch cell first.")
else:
    span_df = pd.DataFrame(collected_span_data)
    evals_dict = create_evaluations_from_span_data(span_df)
    session_eval_df = generate_session_evaluations_from_chain_spans(span_df)

    try:
        from arize.pandas.logger import Client
        client = Client(space_id=space_id, api_key=api_key)
        total_logged = 0
        for name, df in [
            ("ContextRelevance", evals_dict["context_relevance"]),
            ("SyntacticCorrectness", evals_dict["syntactic_correctness"]),
            ("SemanticCorrectness", evals_dict["semantic_correctness"]),
            ("ValidationOutcome", evals_dict["validation_outcome"]),
            ("AgentTrajectory", evals_dict["agent_trajectory"]),
        ]:
            if len(df) == 0:
                continue
            for start in range(0, len(df), CHUNK_SIZE):
                chunk = df.iloc[start : start + CHUNK_SIZE]
                client.log_evaluations_sync(chunk, project_name)
                total_logged += len(chunk)
        if len(session_eval_df) > 0:
            for start in range(0, len(session_eval_df), CHUNK_SIZE):
                chunk = session_eval_df.iloc[start : start + CHUNK_SIZE]
                client.log_evaluations_sync(chunk, project_name)
                total_logged += len(chunk)
        print(f"Logged {total_logged} evaluations to Arize.")
    except Exception as e:
        print(f"Error logging: {e}")

  arize.utils.logging | WARNING | ⚠️ Only 8 out of 500 evaluation data were logged, and 492 of data were not logged successfully for model 'nlp_to_sql_test1'.
  arize.utils.logging | WARNING | ⚠️ Only 6 out of 300 evaluation data were logged, and 294 of data were not logged successfully for model 'nlp_to_sql_test1'.
  arize.utils.logging | WARNING | ⚠️ Only 10 out of 500 evaluation data were logged, and 490 of data were not logged successfully for model 'nlp_to_sql_test1'.
  arize.utils.logging | WARNING | ⚠️ Only 9 out of 500 evaluation data were logged, and 491 of data were not logged successfully for model 'nlp_to_sql_test1'.
  arize.utils.logging | WARNING | ⚠️ Only 25 out of 288 evaluation data were logged, and 263 of data were not logged successfully for model 'nlp_to_sql_test1'.
  arize.utils.logging | WARNING | ⚠️ Only 15 out of 500 evaluation data were logged, and 485 of data were not logged successfully for model 'nlp_to_sql_test1'.
  arize.utils.logging | WARNING | ⚠️ Only 1